In [3]:
import pandas as pd
import glob
import os
from IPython.display import display

path = "../../data/processed/*.csv"
files = glob.glob(path)

summary_stats = []

for file in files:
    filename = os.path.basename(file)
    try:
        df = pd.read_csv(file, low_memory=False)
        total_rows = len(df)
        
        # Smart search for description column
        desc_col = next((c for c in df.columns if c.lower() in ['description', 'job_description', 'text', 'full_text', 'jobposting']), None)
        if desc_col:
            missing_desc = df[desc_col].isna().sum()
        else:
            missing_desc = total_rows 
            
        # Smart search for date column + timezone fix (utc=True)
        date_col = next((c for c in df.columns if 'date' in c.lower() or c.lower() == 'timestamp'), None)
        if date_col:
            dates = pd.to_datetime(df[date_col], errors='coerce', utc=True).dropna()
            if not dates.empty:
                min_date = dates.min().strftime('%Y-%m-%d')
                max_date = dates.max().strftime('%Y-%m-%d')
            else:
                min_date, max_date = "N/A", "N/A"
        else:
            min_date, max_date = "N/A", "N/A"
            
        clean_source = filename.replace('_jobs_dedup.csv', '').replace('_cleaned.csv', '').replace('_jobs', '')
        
        summary_stats.append({
            "Source": clean_source,
            "Total Vacancies": total_rows,
            "Valid Descriptions": total_rows - missing_desc,
            "Missing Desc (%)": round((missing_desc / total_rows) * 100, 2) if total_rows > 0 else 0.0,
            "Oldest Record": min_date,
            "Newest Record": max_date
        })
        
    except Exception as e:
        print(f"Error processing {filename}: {e}")

overview_df = pd.DataFrame(summary_stats)

if not overview_df.empty:
    overview_df = overview_df.sort_values(by="Total Vacancies", ascending=False).reset_index(drop=True)
    total_vacs = overview_df['Total Vacancies'].sum()
    print(f"TOTAL DATASET VOLUME: {total_vacs:,} vacancies\n")
    display(overview_df)
else:
    print("No CSV files found or processed.")

/tmp/ipykernel_3711/719039218.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[date_col], errors='coerce', utc=True).dropna()


TOTAL DATASET VOLUME: 676,235 vacancies



,Source,Total Vacancies,Valid Descriptions,Missing Desc (%),Oldest Record,Newest Record
0,stackoverflow,258009,257590,0.16,2018-03-07,2022-03-31
1,efinancialcareers,208348,157719,24.30,2018-02-27,2024-12-18
2,stepstone_de,112123,23783,78.79,2015-12-18,2017-11-16
3,pracuj_pl,48361,17868,63.05,2015-01-07,2024-12-27
4,ch_data,23474,23474,0.00,2010-09-01,2024-12-28
5,dice_com,20845,20843,0.01,2023-05-19,2024-05-21
6,monster_com,5075,5075,0.00,2018-07-23,2021-12-09
